Instalación de Librerías y Configuración Inicial

In [2]:
%pip install --upgrade --quiet langchain openai memory_profiler pandas numpy matplotlib seaborn google-genai

Note: you may need to restart the kernel to use updated packages.


 Configuración del Entorno y Modelos

In [ ]:
import os
import time
import subprocess
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from memory_profiler import memory_usage
from openai import OpenAI
from google import genai
from google.genai import types

# Configurar modelos disponibles
llm_models = ["gpt-4o", "deepseek/deepseek-r1", "gemini-2.0-flash-thinking-exp-01-21"]
temperature = 0
time_limit = 120  # Tiempo máximo de ejecución

# Configurar directorios de salida
output_directory = f"output2/temperature-{temperature}/"
results_directory = f"results2/temperature-{temperature}/"
Path(output_directory).mkdir(parents=True, exist_ok=True)
Path(results_directory).mkdir(parents=True, exist_ok=True)

# Claves API (asegúrate de configurar las variables de entorno)
openai_api_key = os.getenv("OPENAI_API_KEY")
deepSeek_api_key = os.getenv("DeepSeek_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

# Crear clientes de API para diferentes modelos
client_openai = OpenAI(api_key=openai_api_key)
client_deepseek = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=deepSeek_api_key)
client_gemini = genai.Client(api_key=gemini_api_key)


In [ ]:
def generate_with_gemini(prompt):
    response = client_gemini.models.generate_content(
        model='gemini-2.0-flash-thinking-exp-01-21',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature
        )
    )
    return response.text


Ejecución y Evaluación del Código Generado

In [ ]:
def execute_code(script_path):
    """
    Ejecuta el código generado por el modelo y captura la salida.
    Retorna el resultado de ejecución.
    """
    try:
        proceso = subprocess.Popen(["python", script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        salida, error = proceso.communicate(timeout=time_limit)
        return salida.decode().strip(), error.decode().strip() if error else "No Error"
    
    except subprocess.TimeoutExpired:
        proceso.kill()
        return "TIMEOUT", "TIMEOUT"


Medición del Tiempo de Ejecución

In [ ]:
def execute_with_timer(script_path):
    """
    Ejecuta el código y mide el tiempo de ejecución.
    """
    start_time = time.time()
    result, error = execute_code(script_path)
    execution_time = time.time() - start_time
    return result, error, execution_time


4️⃣ Medición del Uso de Memoria

In [ ]:
def execute_with_memory(script_path):
    """
    Mide el consumo de memoria del script generado.
    """
    def run_script():
        execute_code(script_path)

    mem_usage = memory_usage(run_script, interval=0.1)
    return max(mem_usage)  # Retorna el pico de memoria usado


📌 5️⃣ Estimación de Complejidad Algorítmica

In [ ]:
def estimate_complexity(code):
    """
    Analiza la complejidad del código en términos de estructuras de control.
    """
    try:
        tree = ast.parse(code)
        loops = sum(isinstance(node, (ast.For, ast.While)) for node in ast.walk(tree))
        recursion = sum(isinstance(node, ast.Call) and isinstance(node.func, ast.Name) for node in ast.walk(tree))

        if recursion > 0:
            return "O(n) or worse (Recursion detected)"
        elif loops == 1:
            return "O(n)"
        elif loops > 1:
            return "O(n^2) or worse (Nested loops detected)"
        else:
            return "O(1) (No loops detected)"
    
    except Exception as e:
        return f"Error analyzing complexity: {str(e)}"


6️⃣ Evaluación de Sensibilidad al Prompt

In [ ]:
def evaluate_prompt_variability(model, base_prompt):
    """
    Evalúa cómo varía la respuesta del modelo ante cambios en el prompt.
    """
    variations = [
        base_prompt,
        base_prompt.replace("solve this problem", "generate a Python function to solve"),
        base_prompt + " Ensure the solution is optimized for efficiency."
    ]
    
    results = []
    for prompt in variations:
        response = model(prompt)
        results.append(response)
    
    return results


7️⃣ Medición de Tokens Utilizados

In [ ]:
def count_tokens(model, prompt):
    """
    Cuenta cuántos tokens usa el modelo para responder.
    """
    response = model(prompt, return_token_count=True)
    return response["token_count"]


In [ ]:
8️⃣ Evaluación del Éxito por Dificultad

In [ ]:
def categorize_difficulty(problem_id):
    """
    Clasifica los problemas en tres niveles de dificultad.
    """
    easy_problems = [1, 2, 3, 4, 5]  
    medium_problems = [6, 7, 8, 9, 10]  
    hard_problems = [11, 12, 13, 14, 15]  

    if problem_id in easy_problems:
        return "Easy"
    elif problem_id in medium_problems:
        return "Medium"
    elif problem_id in hard_problems:
        return "Hard"
    else:
        return "Unknown"


9 Generar Resultados y Exportar a CSV

In [ ]:
# DataFrame para almacenar los resultados
df_results = pd.DataFrame(columns=["Model", "Problem ID", "Execution Time", "Memory Usage", 
                                   "Complexity", "Tokens Used", "Correctness", "Prompt Sensitivity"])

def evaluate_model(model_name, script_path, problem_id, prompt):
    """
    Evalúa el modelo en múltiples métricas y almacena los resultados.
    """
    result, error, exec_time = execute_with_timer(script_path)
    memory_used = execute_with_memory(script_path)
    complexity = estimate_complexity(open(script_path).read())
    token_count = count_tokens(model_name, prompt)
    difficulty = categorize_difficulty(problem_id)

    # Almacenar resultado en el DataFrame
    df_results.loc[len(df_results)] = [model_name, problem_id, exec_time, memory_used, complexity, 
                                       token_count, "Success" if "True" in result else "Fail", difficulty]

# Exportar resultados
df_results.to_csv(f"{results_directory}model_evaluation_results.csv", index=False)
print("Resultados guardados en CSV.")


Visualización de los Resultados

In [ ]:
# Cargar los resultados
df_results = pd.read_csv(f"{results_directory}model_evaluation_results.csv")

# Estilo gráfico
sns.set(style="whitegrid")

# Gráfico de tiempo de ejecución
plt.figure(figsize=(10, 5))
sns.boxplot(x="Model", y="Execution Time", data=df_results, palette="coolwarm")
plt.title("Tiempo de Ejecución por Modelo")
plt.xlabel("Modelo")
plt.ylabel("Tiempo de Ejecución (s)")
plt.xticks(rotation=45)
plt.show()
